In [56]:
import pandas as pd
import numpy as np
import tensorflow as tf


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [57]:
# 1. 데이터 불러오기

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/diabetes.csv")

print(df.head())
print(df.shape)

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
(768, 9)


In [58]:
# 2. 입력 / 정답 분리
# BMI가 0인 결측치 행 제거(성능을 높이기 위함)
df = df[df["BMI"] > 0]
X = df.drop("BMI", axis=1).values
y = df["BMI"].values
#회귀는 연속적인 값을 예측하므로 y값 원핫인코딩 안함

In [59]:
# 3. 학습 / 테스트 데이터 분리

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# BMI(y)가 연속형 수치 데이터이므로 클래스 비율 기반의 층화 추출을 적용할 수 없기 때문에 stratify=y 제거

In [60]:
# 4. 데이터 정규화

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [61]:
# 5. 모델 생성

model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(32, activation="relu", input_shape=(8,)))
model.add(tf.keras.layers.Dense(16, activation="relu"))
model.add(tf.keras.layers.Dense(1))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [62]:
# 6. 모델 설정

model.compile(optimizer="adam", loss="mse", metrics=["mae"])

In [63]:
# 7. 모델 구조 출력

model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 833 (3.25 KB)

 Trainable params: 833 (3.25 KB)

 Non-trainable params: 0 (0.00 B)

In [64]:
# 8. 모델 학습

history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

Epoch 1/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 1143.4896 - mae: 33.0891 - val_loss: 1069.5656 - val_mae: 31.9205
Epoch 2/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1119.0021 - mae: 32.7224 - val_loss: 1049.9800 - val_mae: 31.6156
Epoch 3/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1097.8612 - mae: 32.4027 - val_loss: 1031.1698 - val_mae: 31.3188
Epoch 4/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1074.4237 - mae: 32.0482 - val_loss: 1007.9080 - val_mae: 30.9482
Epoch 5/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1044.8523 - mae: 31.5974 - val_loss: 978.5225 - val_mae: 30.4736
Epoch 6/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1008.4305 - mae: 31.0289 - val_loss: 941.5172 - val_mae: 29.8673
Epoch 7/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 962.7809 - mae: 30.2963 - val_loss: 895.7469 - val_mae: 29.1017
Epoch 8/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 907.4014 - mae: 29.3907 - val_loss: 841.4865 - val_mae: 28.1649
Epoch 9/50
16

In [65]:
# 9. 테스트 데이터 예측

y_pred = model.predict(X_test).flatten()

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


In [66]:
# 10. 회귀 성능 평가
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- 회귀 모델 평가 결과 ---")
print(f"MAE (평균 절대 오차) : {mae:.4f}")
print(f"MSE (평균 제곱 오차) : {mse:.4f}")
print(f"R² Score (결정계수) : {r2:.4f}")


--- 회귀 모델 평가 결과 ---
MAE (평균 절대 오차) : 4.8949
MSE (평균 제곱 오차) : 39.0453
R² Score (결정계수) : 0.0383
